# 02 — Filtros de imagem

Filtros espaciais reutilizáveis: convolução, mediana, kernels e gradiente Sobel.

**Dependências:** `numpy`; `skimage` apenas em `equalizar_histograma`.

**Nota:** implementações manuais alinhadas com os Worksheets da unidade curricular.


## Importações


In [ ]:
import numpy as np


## Convolução 2D


In [ ]:
def filtro_convolucao(img, kernel):
    """Convolução com padding 'edge'; saída uint8 em [0, 255]."""
    if kernel.shape[0] != kernel.shape[1]:
        raise ValueError("O kernel deve ser quadrado.")
    s = kernel.shape[0]
    if s % 2 == 0:
        raise ValueError("O tamanho do kernel deve ser ímpar.")
    p = s // 2
    img_p = np.pad(img.astype(np.float64), p, mode="edge")
    h, w = img.shape
    saida = np.zeros((h, w), dtype=np.float64)
    for i in range(p, p + h):
        for j in range(p, p + w):
            viz = img_p[i - p : i + p + 1, j - p : j + p + 1]
            saida[i - p, j - p] = np.sum(viz * kernel)
    return np.clip(saida, 0, 255).astype(np.uint8)


## Filtro da mediana


In [ ]:
def filtro_mediana(img, tamanho_vizinhanca):
    """Mediana em janela quadrada ímpar (implementação manual)."""
    if tamanho_vizinhanca % 2 == 0:
        raise ValueError("O tamanho da vizinhança deve ser ímpar.")
    p = tamanho_vizinhanca // 2
    img_p = np.pad(img, p, mode="edge")
    h, w = img.shape
    saida = np.zeros((h, w), dtype=np.uint8)
    for i in range(p, p + h):
        for j in range(p, p + w):
            viz = img_p[i - p : i + p + 1, j - p : j + p + 1]
            saida[i - p, j - p] = int(np.median(viz))
    return saida


## Construção de kernels


In [ ]:
def kernel_media(tamanho):
    """Kernel de média normalizado."""
    if tamanho % 2 == 0:
        raise ValueError("O tamanho do kernel deve ser ímpar.")
    k = np.ones((tamanho, tamanho), dtype=np.float64)
    return k / (tamanho * tamanho)


def kernel_gaussiano(tamanho, sigma):
    """Kernel Gaussiano 2D normalizado."""
    if tamanho % 2 == 0:
        raise ValueError("O tamanho do kernel deve ser ímpar.")
    if sigma <= 0:
        raise ValueError("Sigma deve ser positivo.")
    w = tamanho // 2
    yy, xx = np.mgrid[-w : w + 1, -w : w + 1]
    g = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return g / g.sum()


def kernels_sobel():
    """Devolve (sobel_x, sobel_y)."""
    sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
    sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float64)
    return sobel_x, sobel_y


## Gradiente Sobel


In [ ]:
def gradiente_sobel(img):
    """Magnitude do gradiente Sobel normalizada para uint8."""
    sobel_x, sobel_y = kernels_sobel()
    gx = filtro_convolucao(img, sobel_x).astype(np.float64)
    gy = filtro_convolucao(img, sobel_y).astype(np.float64)
    magnitude = np.sqrt(gx ** 2 + gy ** 2)
    if magnitude.max() > 0:
        magnitude = (magnitude / magnitude.max()) * 255.0
    return magnitude.astype(np.uint8)


## Equalização de histograma


In [ ]:
def equalizar_histograma(img):
    """Equalização via skimage.exposure; saída uint8."""
    from skimage import exposure

    if img.dtype == np.uint8:
        entrada = img.astype(np.float64) / 255.0
    else:
        entrada = img.astype(np.float64)
        if entrada.max() > 1.0:
            entrada = entrada / entrada.max()
    eq = exposure.equalize_hist(entrada)
    return (np.clip(eq, 0, 1) * 255).astype(np.uint8)
